Make a heatmap of activity in the few second preceding the homing, sorted by pref tuning in the homing

In [1]:
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip3_18mar, JAL6_flip5_25mar, # (unmatched number of neurons and cluster ids) # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_14may, JAL8_flip4_10may]


session_names = ["JAL4_3rdSept","JAL4_19thSept","JAL4_28aug","JAL4_11thSept",
    "JAL5_8thSept","JAL5_21stSept",
    "JAL6_28mar", "JAL6_flip4_21mar", "JAL6_flip3_18mar", "JAL6_flip5_25mar", 
    "JAL7_sesh8_9apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_sesh9_16apr", "JAL7_23apr",
    "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip3_7may",  "JAL8_14may", "JAL8_flip4_10may"] 

In [ ]:
%load_ext autoreload
from behave_analysis.utils.creating_directories import make_directory
from behave_analysis.process.process import Process
from JR_test_scripts.escape.functions.escape_utils import load_homing, load_hdir_cells, load_significant_cells
import matplotlib.gridspec as gridspec
from JR_test_scripts.FigureSaver import Figure_Saver

import os
import numpy as np
import matplotlib.pyplot as plt
import polars as pl
import pandas as pd
from loguru import logger
import dill as pickle
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
from scipy.stats import zscore
import matplotlib.patches as patches
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 20
matplotlib.rcParams['pdf.fonttype'] = 42  # Ensure fonts are embedded in PDF

%matplotlib inline

In [3]:
"""Load the data"""
c_names = ['shelter_only', 'barrier', 'flipped_barrier']
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
explore_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves_explore/")
homie_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
save_path = make_directory("Z:/Jasmine_Laurence/summary_plots/Replay/PreHomingSorted")
tuning_data = '_25bins' # '' or '_50bins' or '_25bins
cond_colors = ['#228B22','#FF8C00','#008B8B']
homing_color = '#6A0DAD'
escape_color = '#E63946'

In [4]:
"""Load data and Extract the homing and escape x and y positions"""

def load_data(exp):
    # load session
    session = Process(exp).load_session()
    base_path = os.path.join(session.base_path, session.processed_path)

    # matrix
    frame_by_cluster_matrix = np.load(
        os.path.join(session.base_path, session.processed_path)
        + "\\"
        + "frame_by_good_cluster_matrix.npy"
    )

    # full x and y pos
    video_df = pl.read_csv(os.path.join(base_path, "full_video_dataframe.csv"))
    y_pos = video_df["mouse_y_position"].to_numpy()
    x_pos = video_df["mouse_x_position"].to_numpy()
    bar = video_df["barrier_present"].to_numpy()
    barflip = video_df["barrier_flipped"].to_numpy()

    # load homings
    h_on, h_off, homing_bool = load_homing(session, int(np.amax(np.unique(video_df['frames'].to_numpy()))))

    # booleans for homing+escape and explore
    # TODO this is a hack into the escapes to remove the 1s of pause before they start running
    escape = video_df['EscapePeriod'].to_numpy()
    escape_bool = np.full_like(escape, False)
    start = np.where(np.diff(escape.astype(int)) == 1)[0]
    end = np.where(np.diff(escape.astype(int)) == -1)[0]
    for s, e in zip(start, end):
        escape_bool[s+40:e] = True

    h_e_bool = (homing_bool | escape_bool) & (video_df['OutofshelterIdx'].to_numpy())
    full_e_bool = (escape_bool) & (video_df['OutofshelterIdx'].to_numpy())
    cond = np.zeros(len(bar))
    cond[bar] += 1
    cond[barflip] += 1

    X = x_pos[h_e_bool]
    Y = y_pos[h_e_bool]
    h_cond = cond[h_e_bool]

    # cell tuning pref
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + 'escape'
    data = np.load(homie_path + exp_nickname + '_ProperTuning' + tuning_data + '.npz')
    preferred_tuning = data['params_full'][:,:,1] #neurons x conditions
    tuning_curve = data['fr_full'] # condition x cell x time

    return frame_by_cluster_matrix, X, Y, h_cond, cond, x_pos, y_pos, bar, barflip, h_e_bool, full_e_bool, preferred_tuning, tuning_curve, session, base_path, video_df, h_on, h_off, homing_bool, escape_bool, escape, cond_colors

In [ ]:
"""Find the homing starts and ends
Subselect complete homings"""

def process_homings(h_e_bool, full_e_bool, h_cond, X, Y, time):
    # starts of homings & escapes
    starts = np.where(np.diff(h_e_bool.astype(int)) > 0)[0] + 1
    ends = np.where(np.diff(h_e_bool.astype(int)) < 0)[0] + 1
    homie_lengths = ends - starts
    counter, h_start, e_start = 0, np.full(len(homie_lengths), 0), np.full(len(homie_lengths)+1, False)
    for i, h in enumerate(homie_lengths):
        if full_e_bool[starts[i]]:
            e_start[i] = True # a bool that tells us which one of the h_starts are escapes
        h_start[i] = counter 
        counter += h
    h_start = np.append(h_start, len(h_cond)) # a list of the start indices of homings + escapes in the homing/escape time period
    e_bool = full_e_bool[h_e_bool] # a boolean that tells us which periods of the homing/escape are escapes

    # where did the mouse start and end
    y_start = Y[h_start[:-1]]
    y_end = Y[h_start[1:]-1]
    long_homie_bool = np.full(len(h_start[:-1]), False)
    homie_id = np.full(len(X), 0) # a vector that increases with each homing
    for h, (s,e) in enumerate(zip(y_start, y_end)):
        homie_id[h_start[h]:h_start[h+1]] = h
        if (s < 512) & (e > 700):
            long_homie_bool[h] = True

    # find the times before the homie starts
    homie = np.where(np.diff(h_e_bool.astype(int)) > 0)[0] + 1
    prebool = np.full(len(h_e_bool), False)
    e_long = []
    counter = 0
    for idx, i in enumerate(homie):
        if long_homie_bool[idx]:
            prebool[i-time:i] = True
            counter += 1
            if e_start[idx]:
                e_long.append(counter)

    return prebool, e_long

In [8]:
"""Make a heatmap of the homing data - separate by condition"""

def make_heatmap_prehomie(prebool, e_long, c, fcm, time, session_name, save_path, cond, sig_cells, preferred_tuning, tuning_curve, c_names):
    width = 350 / 20.4 # mm to inches
    height = 200 / 25.4 # mm to inches

    # Define trial start times (Example: 5 trials at arbitrary points)

    # pull out neural activity for this condition
    neural_matrix = fcm[prebool & (cond == c),:] # select the homing and escape periods for the condition
    neural_matrix = neural_matrix[:,sig_cells[:,c]] # select the cells that are sig in the condition
    neural_matrix = zscore(neural_matrix, axis = 0)

    # sort the cells by preferred tuning
    pt = tuning_curve[c,sig_cells[:,c],:] # condition x cell x time
    isort = np.argsort(preferred_tuning[sig_cells[:,c],c])

    # Set up figure with gridspec (one row for XY plots, one for heatmap)
    fig = plt.figure(figsize=(width, height))
    gs = gridspec.GridSpec(nrows=2, ncols=2, width_ratios = [0.1, 1])

    # plot the tuning curves
    ax = fig.add_subplot(gs[0, 0])
    ax.imshow(zscore(pt[isort,:], axis = 1), vmin=-.5, vmax=1.5, cmap="gray_r", aspect="auto", interpolation="none",
              extent = (0, 100, 0, len(isort)))
    ax.set_ylabel("Sig. tuned Neurons")
    ax.set_xlabel(r'% escape')

    # Create the heatmap
    ax_heatmap = fig.add_subplot(gs[0, 1])
    ax_heatmap.imshow(neural_matrix[:,isort].T, vmin=-.5, vmax=1.5, cmap="gray_r", aspect="auto", interpolation="none",
                        extent=(0, neural_matrix.shape[0]/40, 0, neural_matrix.shape[1]))
    ax_heatmap.set_title(f"Neural activty 3s before homings")
    ax_heatmap.set_xlabel('Time (s)')

    h_start = np.arange(0, np.sum(prebool & (cond == c))/40, time)
    for start, i in enumerate(h_start):    
        # Set limits and plot XY trajectory
        if start in e_long:
            # Add vertical lines for trial starts
            ax_heatmap.axvline(x=i, color = escape_color, linestyle='--')
        else:
            ax_heatmap.axvline(x=i, color=homing_color, linestyle='--')

    # add heatmap of neurons that aer not significantly tuned!
    neural_matrix = fcm[prebool & (cond == c),:]
    neural_matrix = neural_matrix[:,~sig_cells[:,c]] # select the cells that are sig in the condition
    neural_matrix = zscore(neural_matrix, axis = 0)
    pt = tuning_curve[c,~sig_cells[:,c],:] # condition x cell x time
    isort = np.argsort(preferred_tuning[~sig_cells[:,c],c])

    # plot the tuning curves
    ax = fig.add_subplot(gs[1, 0])
    ax.imshow(zscore(pt[isort,:], axis = 1), vmin=-.5, vmax=1.5, cmap="gray_r", aspect="auto", interpolation="none",
              extent = (0, 100, 0, len(isort)))
    ax.set_ylabel("Untuned Neurons")
    ax.set_xlabel(r'% escape')
    
    ax_heatmap = fig.add_subplot(gs[1, 1])
    ax_heatmap.imshow(neural_matrix[:,isort].T, vmin=-.5, vmax=1.5, cmap="gray_r", aspect="auto", interpolation="none",
                        extent=(0, neural_matrix.shape[0]/40, 0, neural_matrix.shape[1]))
    ax_heatmap.set_xlabel('Time (s)')

    h_start = np.arange(0, np.sum(prebool & (cond == c))/40, time)
    for start, i in enumerate(h_start):    
        ax_heatmap.axvline(x=i, color=homing_color, linestyle='--')

    plt.tight_layout()

    # Initialize the Figure_Saver
    fig.savefig(os.path.join(save_path, session_name), dpi=300)

    plt.close(fig)  # Close the figure to free memory

In [ ]:
"""Load data and Extract the homing and escape x and y positions"""
long_homings = True
time = 3 # seconds before the homing start to plot the heatmap
experiments_objects = experiments_objects[8:]
session_names = session_names[8:]
for e, exp in enumerate(experiments_objects):
    session_name = session_names[e]
    fcm, X, Y, h_cond, cond, x_pos, y_pos, bar, barflip, h_e_bool, full_e_bool, preferred_tuning, tuning_curve, session, base_path, video_df, h_on, h_off, homing_bool, escape_bool, escape, cond_colors = load_data(exp)
    sig_cells = load_significant_cells(exp, case = "either_tuned", tuning_data = tuning_data)
    hdir = load_hdir_cells([exp], [session_name])
    sig_cells[hdir,:] = np.full(3, False)

    # filter only the cells that are not hdir and are sig in at least one condition to either dist or escape
    fcm = gaussian_filter1d(fcm, 2, axis = 0)

    prebool, e_long = process_homings(h_e_bool, full_e_bool, h_cond, X, Y, time*40)
    for c, cc in enumerate(conditions):
        save_name = exp.nick_name + '_' + exp.experiment_date + '_Prehoming_sorted_activity_' + cc
        make_heatmap_prehomie(prebool, e_long, c, fcm, time, save_name, save_path, cond, sig_cells, preferred_tuning, tuning_curve, c_names)   

2025-04-30 16:39:56.889 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2025-04-30 16:40:36.104 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...
2025-04-30 16:40:38.881 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2025-04-30 16:40:54.264 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2025-04-30 16:41:27.110 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...
2025-04-30 16:41:31.431 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2025-04-30 16:41:41.620 | INFO     | behave_analysis.process.process:loa